# Olist Brazilian E-Commerce — Apache Spark Data Integration Pipeline

**Dataset:** https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce  
**Environment:** Google Colab / Local PySpark  
**Phases:** 4 (Ingestion) → 5 (Cleaning) → 6 (Integration) → 7 (ETL) → 8 (Analytics) → 9 (Visualisation)

## Phase 4 — Data Ingestion

In [ ]:
# CELL 1: Environment Setup
!pip install pyspark --quiet

import pyspark
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType,
    IntegerType, DoubleType, TimestampType
)

spark = SparkSession.builder \
    .appName("Olist_Brazilian_Ecommerce_Pipeline") \
    .master("local[*]") \
    .config("spark.sql.shuffle.partitions", "8") \
    .config("spark.driver.memory", "4g") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")
print(f"PySpark version : {pyspark.__version__}")
print(f"Spark app       : {spark.sparkContext.appName}")
print(f"Active cores    : {spark.sparkContext.defaultParallelism}")

In [ ]:
# CELL 2: Mount Google Drive and define data paths
from google.colab import drive
drive.mount("/content/drive")

BASE_PATH = "/content/drive/MyDrive/olist_data/raw"

FILE_PATHS = {
    "orders"      : f"{BASE_PATH}/olist_orders_dataset.csv",
    "order_items" : f"{BASE_PATH}/olist_order_items_dataset.csv",
    "customers"   : f"{BASE_PATH}/olist_customers_dataset.csv",
    "products"    : f"{BASE_PATH}/olist_products_dataset.csv",
    "sellers"     : f"{BASE_PATH}/olist_sellers_dataset.csv",
    "payments"    : f"{BASE_PATH}/olist_order_payments_dataset.csv",
    "reviews"     : f"{BASE_PATH}/olist_order_reviews_dataset.csv",
    "geolocation" : f"{BASE_PATH}/olist_geolocation_dataset.csv",
    "category_tr" : f"{BASE_PATH}/product_category_name_translation.csv",
}

TIMESTAMP_FORMAT = "yyyy-MM-dd HH:mm:ss"
VALID_STATUSES = ["delivered","shipped","canceled","unavailable",
                  "invoiced","processing","created","approved"]

BASE_OUTPUT = "/content/drive/MyDrive/olist_data/processed"
VIZ_PATH    = "/content/drive/MyDrive/olist_data/visualisations"

PATHS = {
    "integrated"   : f"{BASE_OUTPUT}/olist_integrated.parquet",
    "aggregated"   : f"{BASE_OUTPUT}/olist_aggregated.parquet",
    "orders_clean" : f"{BASE_OUTPUT}/orders_clean.parquet",
    "items_clean"  : f"{BASE_OUTPUT}/order_items_clean.parquet",
    "reviews_clean": f"{BASE_OUTPUT}/reviews_clean.parquet",
}

import os
os.makedirs(BASE_OUTPUT, exist_ok=True)
os.makedirs(VIZ_PATH,    exist_ok=True)
print("Paths configured.")

In [ ]:
# CELL 3: Explicit schema definitions for all 9 tables
schema_orders = StructType([
    StructField("order_id",                       StringType(),  True),
    StructField("customer_id",                    StringType(),  True),
    StructField("order_status",                   StringType(),  True),
    StructField("order_purchase_timestamp",       StringType(),  True),
    StructField("order_approved_at",              StringType(),  True),
    StructField("order_delivered_carrier_date",   StringType(),  True),
    StructField("order_delivered_customer_date",  StringType(),  True),
    StructField("order_estimated_delivery_date",  StringType(),  True),
])
schema_order_items = StructType([
    StructField("order_id",            StringType(),  True),
    StructField("order_item_id",       IntegerType(), True),
    StructField("product_id",          StringType(),  True),
    StructField("seller_id",           StringType(),  True),
    StructField("shipping_limit_date", StringType(),  True),
    StructField("price",               DoubleType(),  True),
    StructField("freight_value",       DoubleType(),  True),
])
schema_customers = StructType([
    StructField("customer_id",              StringType(), True),
    StructField("customer_unique_id",       StringType(), True),
    StructField("customer_zip_code_prefix", StringType(), True),
    StructField("customer_city",            StringType(), True),
    StructField("customer_state",           StringType(), True),
])
schema_products = StructType([
    StructField("product_id",                 StringType(),  True),
    StructField("product_category_name",      StringType(),  True),
    StructField("product_name_lenght",        IntegerType(), True),
    StructField("product_description_lenght", IntegerType(), True),
    StructField("product_photos_qty",         IntegerType(), True),
    StructField("product_weight_g",           IntegerType(), True),
    StructField("product_length_cm",          IntegerType(), True),
    StructField("product_height_cm",          IntegerType(), True),
    StructField("product_width_cm",           IntegerType(), True),
])
schema_sellers = StructType([
    StructField("seller_id",              StringType(), True),
    StructField("seller_zip_code_prefix", StringType(), True),
    StructField("seller_city",            StringType(), True),
    StructField("seller_state",           StringType(), True),
])
schema_payments = StructType([
    StructField("order_id",             StringType(),  True),
    StructField("payment_sequential",   IntegerType(), True),
    StructField("payment_type",         StringType(),  True),
    StructField("payment_installments", IntegerType(), True),
    StructField("payment_value",        DoubleType(),  True),
])
schema_reviews = StructType([
    StructField("review_id",               StringType(),  True),
    StructField("order_id",                StringType(),  True),
    StructField("review_score",            IntegerType(), True),
    StructField("review_comment_title",    StringType(),  True),
    StructField("review_comment_message",  StringType(),  True),
    StructField("review_creation_date",    StringType(),  True),
    StructField("review_answer_timestamp", StringType(),  True),
])
schema_geolocation = StructType([
    StructField("geolocation_zip_code_prefix", StringType(), True),
    StructField("geolocation_lat",             DoubleType(), True),
    StructField("geolocation_lng",             DoubleType(), True),
    StructField("geolocation_city",            StringType(), True),
    StructField("geolocation_state",           StringType(), True),
])
schema_category_tr = StructType([
    StructField("product_category_name",         StringType(), True),
    StructField("product_category_name_english", StringType(), True),
])
print("All 9 schemas defined.")